In [1]:
!pip install scikit-surprise tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2544606 sha256=568bc4fee4c9d034ffda3d31ceacedca62901c28f604ec3357ad4fb0a03b7382
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [2]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

In [3]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o ml-latest-small.zip

ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')

print("Avaliações:")
print(ratings.head())

print("\nFilmes:")
print(movies.head())

--2025-11-12 04:34:28--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  2.45MB/s    in 0.4s    

2025-11-12 04:34:28 (2.45 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  
Avaliações:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  9649838

In [4]:
user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
similarity_items = user_movie_matrix.corr(min_periods=20)
toy_story_id = movies[movies['title'].str.contains('Toy Story', case=False, regex=False)].iloc[0]['movieId']

print(f"\n🎬 Filmes mais semelhantes a: Toy Story (ID {toy_story_id})")
print(similarity_items[toy_story_id].dropna().sort_values(ascending=False).head(10))


🎬 Filmes mais semelhantes a: Toy Story (ID 1)
movieId
1         1.000000
3114      0.699211
102125    0.674658
95167     0.663129
2699      0.652424
8961      0.643301
6377      0.618701
588       0.611892
3967      0.610921
3868      0.607630
Name: 1, dtype: float64


In [6]:
similarity_users = user_movie_matrix.T.corr(min_periods=20)

def recomendar_por_usuario(user_id, n=5):
    user_ratings = user_movie_matrix.loc[user_id]
    similar_users = similarity_users[user_id].dropna().sort_values(ascending=False)
    similar_users = similar_users.drop(user_id, errors='ignore')  # remove o próprio usuário

    weighted_scores = pd.Series(0, index=user_movie_matrix.columns)
    sum_weights = pd.Series(0, index=user_movie_matrix.columns)

    for other_user, similarity in similar_users.head(10).items():
        other_ratings = user_movie_matrix.loc[other_user]
        mask = other_ratings.notnull()
        weighted_scores[mask] += similarity * other_ratings[mask]
        sum_weights[mask] += similarity

    recommendations = (weighted_scores / sum_weights).sort_values(ascending=False)
    recommendations = recommendations[user_ratings.isnull()].head(n)

    return recommendations

print("\n🎥 Recomendações (User-User CF) para o usuário 42:\n")
print(recomendar_por_usuario(42))



🎥 Recomendações (User-User CF) para o usuário 42:

movieId
2       5.0
6377    5.0
7361    5.0
216     5.0
162     5.0
dtype: float64


/tmp/ipython-input-4284468389.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[3.10683467 3.10683467 3.10683467 3.88354333 3.10683467 3.88354333
 3.88354333 3.10683467 3.88354333 0.77670867 3.10683467 3.88354333
 3.88354333 3.10683467 3.10683467 3.88354333 3.10683467 2.330126
 3.88354333 2.330126   3.88354333 2.330126   3.10683467 3.88354333
 1.55341733 3.10683467 0.77670867 3.10683467 1.55341733 0.77670867
 0.77670867 1.55341733 3.88354333 3.88354333 0.77670867 1.55341733
 3.88354333 0.77670867 3.88354333 3.10683467 3.88354333 3.10683467
 3.88354333 3.10683467 3.88354333 2.330126   0.77670867 3.88354333
 3.10683467 3.88354333 2.330126   3.88354333 3.10683467 2.330126
 3.10683467 2.330126   0.77670867 3.88354333 3.88354333 3.88354333
 3.10683467 0.77670867 3.88354333 3.88354333 3.88354333 0.77670867
 3.10683467 0.77670867 3.10683467 3.10683467 3.88354333 0.77670867
 3.88354333 3.88354333 2.330126   

In [7]:
X = user_movie_matrix.fillna(0).values
n_users, n_movies = X.shape

input_layer = Input(shape=(n_movies,))
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)
decoded = Dense(n_movies, activation='linear')(encoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(X, X, epochs=20, batch_size=32, verbose=0)

X_pred = autoencoder.predict(X)

mask = X > 0
rmse_auto = np.sqrt(mean_squared_error(X[mask], X_pred[mask]))
mae_auto = mean_absolute_error(X[mask], X_pred[mask])

print(f"\n📈 Autoencoder -> RMSE: {rmse_auto:.4f}, MAE: {mae_auto:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

📈 Autoencoder -> RMSE: 2.2719, MAE: 1.8667


In [8]:
resultados = pd.DataFrame({
    'Modelo': [
        'Filtragem Colaborativa (Usuário-Usuário)',
        'Filtragem Colaborativa (Item-Item)',
        'Autoencoder (Deep Learning)'
    ],
    'RMSE': [np.nan, np.nan, rmse_auto],
    'MAE': [np.nan, np.nan, mae_auto]
})

print("\n📊 Comparativo de Desempenho:")
print(resultados)


📊 Comparativo de Desempenho:
                                     Modelo     RMSE       MAE
0  Filtragem Colaborativa (Usuário-Usuário)      NaN       NaN
1        Filtragem Colaborativa (Item-Item)      NaN       NaN
2               Autoencoder (Deep Learning)  2.27187  1.866732
